# 00 — Optuna PostgreSQL Connectivity Check

Notebook opsional untuk memastikan local/shared PostgreSQL Optuna bisa diakses dari laptop/worker.
Ini bukan schema migration orchestration database.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PY_DIR = PROJECT_ROOT / "patched_program_files"
if str(PY_DIR) not in sys.path:
    sys.path.insert(0, str(PY_DIR))

from env_loader import load_dotenv_if_exists, require_env
from optuna_postgres_utils import create_or_load_study,list_studies,delete_study_if_exists,delete_test_studies

load_dotenv_if_exists(PROJECT_ROOT / ".env")
OPTUNA_STORAGE_URL = require_env("OPTUNA_STORAGE_URL")
print(OPTUNA_STORAGE_URL)

In [ ]:
from urllib.parse import urlparse

parsed = urlparse(OPTUNA_STORAGE_URL)

print("DB host:", parsed.hostname)
print("DB port:", parsed.port)
print("DB name:", parsed.path.replace("/", ""))
print("DB user:", parsed.username)

In [ ]:
# Membuat study test kecil. Hapus/abaikan setelah koneksi sukses.
study = create_or_load_study(
    study_name="connection_test_study",
    storage_url=OPTUNA_STORAGE_URL,
    direction="minimize",
    seed=42,
)
trial = study.ask()
x = trial.suggest_float("x", 0.0, 1.0)
study.tell(trial, x)
print("OK. Trial:", trial.number, "value:", x)

In [ ]:
studies = list_studies(storage_url=OPTUNA_STORAGE_URL)
studies

In [ ]:
# Melihat Study yang ingin dihapus
targets = delete_test_studies(
    storage_url=OPTUNA_STORAGE_URL,
    exact_names=["connection_test_study"],
    dry_run=True,
)

targets

In [ ]:
# Menghapus Study yang ingin dihapus dan cek hasilnya

deleted = delete_study_if_exists(
    study_name="connection_test_study",
    storage_url=OPTUNA_STORAGE_URL,
)

print("Deleted:", deleted)